In [ ]:
!pip install swig "gymnasium[box2d]"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class NoisyLinear(nn.Module):
    # y=(mu_w + sigma_w * epsolon_w)x + mu_b + sigma_b * epsilon_b
    def __init__(self, in_features, out_features, std_init=0.1):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(
            torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))

        self.register_buffer(
            'weight_epsilon', torch.empty(out_features, in_features))
        self.register_buffer('bias_epsilon', torch.empty(out_features))

        mu_range = 1 / math.sqrt(in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(std_init / math.sqrt(in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(std_init / math.sqrt(out_features))

        self.reset_noise()
    
    def reset_noise(self,):
        # f(x) = sign(x) * sqrt(|x|)
        f = lambda x: x.sign().mul_(x.abs().sqrt())
        eps_in = f(torch.randn(self.in_features,device=device))
        eps_out = f(torch.randn(self.out_features, device=device))

        self.weight_epsilon.copy_(eps_out.outer(eps_in))
        self.bias_epsilon.copy_(eps_out)
        
    def forward(self, x):
        if self.training:
            weight = self.weight_mu + self.weight_sigma*self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma*self.bias_epsilon
        else: 
            weight, bias = self.weight_mu, self.bias_mu
        return F.linear(x, weight, bias)



class QNetwork(nn.Module):
    def __init__(self, state_size,  action_size, seed):
        super().__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

        self.act = nn.ReLU()
        # code here

    def forward(self, state):
        return self.fc3(self.act(self.fc2(self.act(self.fc1(state)))))


class DuelingQNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed):
        super().__init__()
        self.seed = torch.manual_seed(seed)

        self.feature_extractor = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU()
        )
        self.values = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.advantages = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
        # code here

    def forward(self, state):
        y = self.feature_extractor(state)
        v = self.values(y)
        a = self.advantages(y)
        Q = v + (a - a.mean(dim=1, keepdim=True))
        return Q
    
class DuelingNoisyQNetwork(nn.Module):
    def __init__(self, state_shape, action_size, seed, n_atoms):
        super().__init__()
        self.seed = torch.manual_seed(seed)
        self.n_atoms = n_atoms
        self.action_size = action_size
        channel_in, h,w = state_shape
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=channel_in, out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.values = nn.Sequential(
            NoisyLinear(3136, 256),
            nn.ReLU(),
            NoisyLinear(256, n_atoms)
        )

        self.advantages = nn.Sequential(
            NoisyLinear(3136, 256), 
            nn.ReLU(),
            NoisyLinear(256, action_size * n_atoms)
        )
        
    def forward(self, state):
        state = state.float()/255.0
        y = self.encoder(state)
        v = self.values(y).reshape(-1, 1, self.n_atoms) # (batch_size, 1, 51)
        a = self.advantages(y).reshape(-1, self.action_size, self.n_atoms) # (batch_size, action_size, 51)
        Q = v + (a - a.mean(dim=1, keepdim=True)) # (batch_size, action_size, 51) - логиты распределения для каждого действия
        return Q

    def reset_noise(self):
        self.values[0].reset_noise()
        self.values[2].reset_noise()
        self.advantages[0].reset_noise()
        self.advantages[2].reset_noise()

In [ ]:
import numpy as np
import random
from collections import namedtuple, deque
import torch
import torch.nn.functional as F
import torch.optim as optim

BUFFER_SIZE = int(5e5)
BATCH_SIZE = 256
GAMMA = 0.99
TAU = 5e-3
LR = 1e-4
UPDATE_EVERY = 4
N_ATOMS = 51

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


class Agent():
    def __init__(self, state_size, action_size, n_atoms, seed, n_step=3):
        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(seed)
        self.n_step = n_step
        self.n_step_buffer = deque(maxlen=self.n_step)

        self.n_atoms = n_atoms
        self.vmin = -100
        self.vmax = 1000
        self.dz = (self.vmax-self.vmin) / (self.n_atoms - 1)
        self.supports = torch.linspace(self.vmin, self.vmax, self.n_atoms, device=device)

        # Q-Network
        self.qnetwork_local = DuelingNoisyQNetwork(
            state_size, action_size, seed, n_atoms=n_atoms).to(device)
        self.qnetwork_target = DuelingNoisyQNetwork(
            state_size, action_size, seed, n_atoms=n_atoms).to(device)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=LR)
        # self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='max', patience=50, min_lr=1e-6)
        self.memory = PrioritizingExperienceBuffer(
            self.state_size, self.action_size, BUFFER_SIZE, beta=0.4)
        self.t_step = 0

    def _get_n_step(self):
        state, action = self.n_step_buffer[0][:2]
        reward, next_state, term = self.n_step_buffer[-1][2:]

        for transition in reversed(list(self.n_step_buffer)[:-1]):
            r, s_next, t = transition[2:]
            reward = r + GAMMA*reward*(1-t)
            if t:
                next_state, term = s_next, t

        return state, action, reward, next_state, term

    def step(self, state, action, reward, next_state, done, terminated):
        # multi-step block
        self.n_step_buffer.append((state, action, reward, next_state, terminated))
        if len(self.n_step_buffer) == self.n_step and not done:
            s, a, r, s_next, term = self._get_n_step()
            self.memory.add(s, a, r, s_next, term)

        if done:
            while len(self.n_step_buffer) > 0:
                s, a, r, s_next, term = self._get_n_step()
                self.memory.add(s, a, r, s_next, term)
                self.n_step_buffer.popleft()

        # end multi step block
        self.t_step = (self.t_step + 1) % UPDATE_EVERY
        if self.t_step == 0:
            if self.memory.real_size > 10000:
                experiences, weights, tree_idxs = self.memory.sample(
                    BATCH_SIZE)
                loss=self.learn(experiences, weights, tree_idxs, GAMMA)
                return loss
        return None

    def act(self, state, is_eval=False):  # not using e-greedy
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        if is_eval:
            self.qnetwork_local.eval()
        else:
            self.qnetwork_local.train()


        with torch.no_grad():
            action_values = self.qnetwork_local(state)
            probs = F.softmax(action_values, dim=-1)
        self.qnetwork_local.train()
        # return np.argmax(action_values.cpu().data.numpy())
        q_values = (probs * self.supports).sum(dim=2) # (1, action_size)
        return q_values.squeeze(0).argmax().item()

    def learn(self, experiences, weights, tree_idxs, gamma):
        self.qnetwork_local.reset_noise()
        self.qnetwork_target.reset_noise()
        
        states, actions, rewards, next_states, dones = experiences
        dz = (self.vmax - self.vmin) / (self.n_atoms - 1)
        batch_size = states.size(0)
        
        with torch.no_grad():
            next_logits_local = self.qnetwork_local(next_states)
            next_probs_local = F.softmax(next_logits_local, dim=-1)
            next_action_values = (next_probs_local * self.supports).sum(dim=2)
            
            best_actions = next_action_values.argmax(dim=1).unsqueeze(1).unsqueeze(2)
            best_actions = best_actions.expand(-1, -1, self.n_atoms)
            
            next_logits_target = self.qnetwork_target(next_states)
            next_probs_target = F.softmax(next_logits_target, dim=-1)
            p_next = next_probs_target.gather(1, best_actions).squeeze(1)

            gamma_n = gamma ** self.n_step
            Tz = rewards + gamma_n * self.supports * (1 - dones)
            Tz = Tz.clamp(self.vmin, self.vmax)
            b = (Tz - self.vmin) / dz
            l = b.floor().long()
            u = b.ceil().long()

            dl = u.float() - b
            du = b - l.float()
            dl[(l == u)] = 1.0
            du[(l == u)] = 0.0

            m = torch.zeros(batch_size, self.n_atoms).to(device)

            offset = torch.linspace(0, (batch_size - 1) * self.n_atoms, batch_size).long().unsqueeze(1).expand(batch_size, self.n_atoms).to(device)
            m.view(-1).index_add_(0, (l + offset).view(-1), (p_next * dl).view(-1))
            m.view(-1).index_add_(0, (u + offset).view(-1), (p_next * du).view(-1))

        logits_local = self.qnetwork_local(states)
        log_probs = F.log_softmax(logits_local, dim=-1)
        log_p = log_probs.gather(1, actions.unsqueeze(-1).expand(-1, -1, self.n_atoms)).squeeze(1)
        
        loss = -(m * log_p).sum(dim=1)
        
        elementwise_loss = loss
        loss = (loss * weights.squeeze(1)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(self.qnetwork_local.parameters(), 10.0)
        
        self.optimizer.step()
        
        self.memory.update_priorities(tree_idxs, elementwise_loss.detach().cpu().numpy())
        self.soft_update(self.qnetwork_local, self.qnetwork_target, TAU)
        return loss.item()
    
    def soft_update(self, local_model, target_model, tau):
        '''
        θ_target = τ*θ_local + (1-τ)*θ_target
        '''
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(
                tau*local_param.data + (1.0-tau)*target_param.data)


class PrioritizingExperienceBuffer:
    def __init__(self, state_shape, action_size, buffer_size, eps=0.01, alpha=0.4, beta=0.1, beta_incr=1e-5):
        self.tree = SumTree(size=buffer_size)

        self.eps = eps
        self.alpha = alpha
        self.beta = beta
        self.max_priority = eps
        self.beta_incr = beta_incr

        self.state = torch.empty(buffer_size, *state_shape, dtype=torch.uint8)
        self.action = torch.empty(buffer_size, 1, dtype=torch.long)
        self.reward = torch.empty(buffer_size, dtype=torch.float)
        self.next_state = torch.empty(
            buffer_size, *state_shape, dtype=torch.uint8)
        self.done = torch.empty(buffer_size, dtype=torch.float)

        self.count = 0
        self.real_size = 0
        self.size = buffer_size

    def add(self, state, action, reward, next_state, done):
        self.tree.add(self.max_priority, self.count)

        self.state[self.count] = torch.as_tensor(state)
        self.action[self.count] = torch.as_tensor(action)
        self.reward[self.count] = torch.as_tensor(reward)
        self.next_state[self.count] = torch.as_tensor(next_state)
        self.done[self.count] = torch.as_tensor(done)

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def sample(self, batch_size):
        assert self.real_size >= batch_size
        self.beta = min(1.0, self.beta+self.beta_incr)
        sample_idxs, tree_idxs = [], []
        priorities = torch.empty(batch_size, 1, dtype=torch.float)

        segment = self.tree.total / batch_size
        for i in range(batch_size):
            a, b = segment * i, segment * (i + 1)
            while True:
                cumsum = random.uniform(a, b)
                tree_idx, priority, sample_idx = self.tree.get(cumsum)
                if sample_idx is not None: break
            priorities[i] = float(priority)
            tree_idxs.append(tree_idx)
            sample_idxs.append(sample_idx)

        probs = priorities / self.tree.total
        weights = (self.real_size * probs) ** -self.beta
        weights = (weights / weights.max()).to(device)

        batch = (
            self.state[sample_idxs].to(device),
            self.action[sample_idxs].to(device),
            self.reward[sample_idxs].unsqueeze(1).to(device),
            self.next_state[sample_idxs].to(device),
            self.done[sample_idxs].unsqueeze(1).to(device)
        )
        return batch, weights, tree_idxs

    def update_priorities(self, data_idxs, priorities):
        for data_idx, priority in zip(data_idxs, priorities):
            priority = (priority + self.eps) ** self.alpha
            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

# class ReplayBuffer:
#     def __init__(self, action_size, buffer_size, batch_size, seed):
#         self.action_size = action_size
#         self.memory = deque(maxlen=buffer_size)
#         self.batch_size = batch_size
#         self.seed = random.seed(seed)
#         self.experience = namedtuple('Experience', field_names=[
#                                      'state', 'action', 'reward', 'next_state', 'done'])

#     def add(self, state, action, reward, next_state, done):
#         e = self.experience(state, action, reward, next_state, done)
#         self.memory.append(e)

#     def sample(self):
#         experiences = random.sample(self.memory, k=self.batch_size)
#         states = torch.from_numpy(
#             np.vstack([e.state for e in experiences if e is not None])).float().to(device)
#         actions = torch.from_numpy(
#             np.vstack([e.action for e in experiences if e is not None])).long().to(device)
#         rewards = torch.from_numpy(
#             np.vstack([e.reward for e in experiences if e is not None])).float().to(device)
#         next_states = torch.from_numpy(np.vstack(
#             [e.next_state for e in experiences if e is not None])).float().to(device)
#         dones = torch.from_numpy(
#             np.vstack([e.done for e in experiences if e is not None])).float().to(device)

#         return (states, actions, rewards, next_states, dones)

#     def __len__(self):
#         return len(self.memory)


In [ ]:
class SumTree:
    def __init__(self, size):
        self.nodes = [0] * (2 * size - 1)
        self.data = [None] * size

        self.size = size
        self.count = 0
        self.real_size = 0

    @property
    def total(self):
        return self.nodes[0]

    def update(self, data_idx, value):
        idx = data_idx + self.size - 1
        change = value - self.nodes[idx]

        self.nodes[idx] = value

        parent = (idx - 1) // 2
        while parent >= 0:
            self.nodes[parent] += change
            parent = (parent - 1) // 2

    def add(self, value, data):
        self.data[self.count] = data
        self.update(self.count, value)

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def get(self, cumsum):
        assert cumsum <= self.total

        idx = 0
        while 2 * idx + 1 < len(self.nodes):
            left, right = 2*idx + 1, 2*idx + 2

            if cumsum <= self.nodes[left]:
                idx = left
            else:
                idx = right
                cumsum = cumsum - self.nodes[left]

        data_idx = idx - self.size + 1

        return data_idx, self.nodes[idx], self.data[data_idx]

# Train

In [ ]:
import gymnasium as gym

class SkipFrame(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip=skip

    def step(self, action):
        total_reward=0.0
        terminated=False
        truncated=False

        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            if terminated or truncated: break
        return obs, total_reward, terminated, truncated, info

In [ ]:
import gymnasium as gym
import numpy as np

class CropObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0, high=255, 
            shape=(shape[0] - 12, shape[1], shape[2]), 
            dtype=self.observation_space.dtype
        )

    def observation(self, observation):
        return observation[:-12, :, :]

In [ ]:
import gymnasium as gym
from collections import deque
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
from gymnasium.wrappers import GrayscaleObservation, FrameStackObservation, ResizeObservation

env = gym.make("CarRacing-v3", continuous=False)
env = SkipFrame(env, skip=4)
env = CropObservation(env)
env = GrayscaleObservation(env)
env = ResizeObservation(env, (84,84))
env = FrameStackObservation(env, 4)
state_size = env.observation_space.shape
action_size = env.action_space.n


# agent = Agent(state_size, action_size, n_atoms=N_ATOMS, seed=1)
# pretrained = torch.load('/kaggle/input/datasets/vld5555/finall/final_800600_episodes', map_location=device)
# model_dict = agent.qnetwork_local.state_dict()
# pretrained_dict = {k: v for k, v in pretrained.items() if k in model_dict and v.size() == model_dict[k].size()}
# model_dict.update(pretrained_dict)
# agent.qnetwork_local.load_state_dict(model_dict)
# agent.qnetwork_target.load_state_dict(agent.qnetwork_local.state_dict())

In [ ]:
def train(episodes=800, max_t=1000):
    scores=[]
    scores_window = deque(maxlen=100)


    total_opt_steps=0

    for episode in range(1, episodes+1):
        state, _ = env.reset()
        agent.qnetwork_local.reset_noise()
        score = 0

        for t in range(max_t):
            action = agent.act(state) 
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated 
            loss =agent.step(state, action, reward, next_state, done, terminated)


            
            state=next_state
            score+=reward
            if done: break
        
        scores_window.append(score)
        scores.append(score)

        # if episode>100:
            # agent.scheduler.step(np.mean(scores_window))
        # current_lr = agent.optimizer.param_groups[0]['lr']
        print(f'Episode {episode} Mean reward: {np.mean(scores_window):.2f}')
        
        if episode % 100 == 0:
            print(f'Episode {episode}\nMean reward: {np.mean(scores_window):.2f}')
            torch.save(agent.qnetwork_local.state_dict(), f'с51_model_800+{episode}_episodes')
        
        if np.mean(scores_window) > 900:
            torch.save(agent.qnetwork_local.state_dict(), 'best_model.pth')
            

            
    return scores

start=time.time()

scores = train()
print(f'Train time: {time.time()-start}')
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('reward')
plt.xlabel('episode')
plt.show()

env.close()

In [ ]:
torch.save(agent.qnetwork_local.state_dict(), '800_final_episods')

# Test

In [ ]:
import gymnasium as gym
import torch
import numpy as np
from gymnasium.wrappers import GrayscaleObservation, FrameStackObservation, ResizeObservation



class SkipFrame(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        terminated = False
        truncated = False
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        return obs, total_reward, terminated, truncated, info


class CropObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0, high=255,
            shape=(shape[0] - 12, shape[1], shape[2]),
            dtype=self.observation_space.dtype
        )

    def observation(self, observation):
        return observation[:-12, :, :]


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
env = gym.make('CarRacing-v3', continuous=False, render_mode='human')
env = SkipFrame(env)
env = CropObservation(env)
env = GrayscaleObservation(env)
env = ResizeObservation(env, (84, 84))
env = FrameStackObservation(env, 4)
state_size = env.observation_space.shape
action_size = env.action_space.n

agent = Agent(state_size=state_size, action_size=action_size, n_atoms=N_ATOMS, seed=1)

weights = torch.load('/Users/vladharcenko/Desktop/ML/РСК/task5/rainbow_weights',
                     map_location=device, weights_only=True)
agent.qnetwork_local.load_state_dict(weights)

agent.qnetwork_local.eval()

num_episodes = 5
scores = []


for i in range(1, num_episodes + 1):
    state, _ = env.reset()
    score = 0
    done = False

    while not done:
        action = agent.act(state, is_eval=True)

        state, reward, terminated, truncated, _ = env.step(action)
        score += reward
        done = terminated or truncated

    scores.append(score)
    print(f"{i}/{num_episodes} episode. Score {score:.2f}")

env.close()

print(f"Mean reward: {np.mean(scores):.2f}")
print(f"max reward {np.max(scores):.2f}")
print(f"Min reward {np.min(scores):.2f}")


In [ ]:
import gymnasium as gym
import imageio
env = gym.make('CarRacing-v3', render_mode='rgb_array')
obs, _ = env.reset()
frames = []
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    print(obs, reward, terminated, truncated, _)
            # obs — RGB frame
    done = terminated or truncated
    break
env.close()